# **Threads Scraper Notebook for HealthPH+**


# **Dependencies**

In [1]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [2]:
from pathlib import Path
import sys


In [3]:
print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## Threads

In [4]:
# Threads scraper config (Playwright, free/public mode)
TH_PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
        if (candidate / "docs" / "keywords").exists()
    ),
    Path.cwd(),
)
TH_KEYWORDS_DIR = TH_PROJECT_ROOT / "docs" / "keywords"
TH_KEYWORD_FILES = [
    #TH_KEYWORDS_DIR / "covid_keywords.csv",
    TH_KEYWORDS_DIR / "covid_keywords2.csv",
    #TH_KEYWORDS_DIR / "ri_keywords.csv",
    TH_KEYWORDS_DIR / "ri_keywords2.csv",
    #TH_KEYWORDS_DIR / "tb_keywords.csv",
    TH_KEYWORDS_DIR / "tb_keywords2.csv",
]
TH_KEYWORD_COLUMNS = [None] * len(TH_KEYWORD_FILES)  # None means first column of each CSV

# Fast profile for notebook runs. Set to False for full collection.
TH_FAST_MODE = True
TH_MAX_KEYWORDS = 30 if TH_FAST_MODE else None
TH_MAX_POSTS_PER_KEYWORD = 50
TH_SCROLL_ROUNDS = 6 if TH_FAST_MODE else 20
TH_SCROLL_WAIT_MS = 700 if TH_FAST_MODE else 1500
TH_HEADLESS = True
TH_NAV_TIMEOUT_MS = 15000 if TH_FAST_MODE else 60000

TH_START_DATE = pd.Timestamp("2025-01-01", tz="UTC")
TH_END_DATE = pd.Timestamp.now(tz="UTC")

TH_OUTPUT_DIR = TH_PROJECT_ROOT / "data" / "raw" / "threads"
TH_OUTPUT_FILE = TH_OUTPUT_DIR / "threads_notebook.csv"

print("Keyword files:", TH_KEYWORD_FILES)
print("Fast mode:", TH_FAST_MODE)
print("Max keywords:", TH_MAX_KEYWORDS)
print("Date range:", TH_START_DATE, "to", TH_END_DATE)
print("Output (append mode):", TH_OUTPUT_FILE)


Keyword files: [PosixPath('/Users/angelodelapaz/Documents/GitHub/healthphpersonal/docs/keywords/covid_keywords2.csv'), PosixPath('/Users/angelodelapaz/Documents/GitHub/healthphpersonal/docs/keywords/ri_keywords2.csv'), PosixPath('/Users/angelodelapaz/Documents/GitHub/healthphpersonal/docs/keywords/tb_keywords2.csv')]
Fast mode: True
Max keywords: 30
Date range: 2025-01-01 00:00:00+00:00 to 2026-03-11 02:02:54.983841+00:00
Output (append mode): /Users/angelodelapaz/Documents/GitHub/healthphpersonal/data/raw/threads/threads_notebook.csv


In [5]:
from typing import Any
from urllib.parse import quote_plus
import hashlib
import re

try:
    from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc


def _th_safe_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def _th_normalize_ws(text: str) -> str:
    return re.sub(r"\\s+", " ", _th_safe_text(text)).strip()


def _th_to_utc_datetime(value: Any) -> pd.Timestamp | None:
    if value is None or value == "":
        return None

    raw = _th_safe_text(value)
    if raw.isdigit():
        number = int(raw)
        unit = "s" if number <= 9999999999 else "ms"
        ts = pd.to_datetime(number, unit=unit, errors="coerce", utc=True)
    else:
        ts = pd.to_datetime(raw, errors="coerce", utc=True)

    if pd.isna(ts):
        return None
    return ts


def _th_to_unix_seconds(ts: pd.Timestamp | None) -> str:
    if ts is None:
        return ""
    return str(int(ts.timestamp()))


def _th_parse_compact_int(value: str) -> int | None:
    text = _th_safe_text(value).replace(",", "")
    if not text:
        return None

    match = re.search(r"(\\d+(?:\\.\\d+)?)\\s*([KMB])?", text, flags=re.IGNORECASE)
    if not match:
        return None

    number = float(match.group(1))
    suffix = (match.group(2) or "").upper()
    multiplier = {"": 1, "K": 1000, "M": 1_000_000, "B": 1_000_000_000}.get(suffix, 1)
    return int(number * multiplier)


def _th_extract_metric(text: str, labels: tuple[str, ...]) -> int | None:
    lowered = _th_safe_text(text).lower()
    if not lowered:
        return None

    for label in labels:
        pattern = rf"(\\d[\\d,]*(?:\\.\\d+)?\\s*[kmb]?)\\s*{label}"
        found = re.search(pattern, lowered, flags=re.IGNORECASE)
        if found:
            return _th_parse_compact_int(found.group(1))
    return None


def _th_extract_username(url: str) -> str:
    match = re.search(r"threads\\.com/@([^/]+)/post/", url)
    return match.group(1) if match else ""


def _th_extract_post_id(url: str, text: str, created_at: str) -> str:
    numeric_match = re.search(r"/(\\d{8,})/?$", url)
    if numeric_match:
        return numeric_match.group(1)

    base = f"{url}|{created_at}|{text[:120]}"
    return hashlib.sha1(base.encode("utf-8")).hexdigest()


def _th_load_keywords_from_files(keyword_files: list[Path], keyword_columns: list[Any] | None = None) -> list[str]:
    if keyword_columns is None:
        keyword_columns = [None] * len(keyword_files)

    if len(keyword_columns) != len(keyword_files):
        raise ValueError("keyword_columns must have the same length as keyword_files")

    loaded: list[str] = []
    seen: set[str] = set()

    for file_path, column in zip(keyword_files, keyword_columns):
        path = Path(file_path)
        if not path.exists():
            print(f"Warning: keyword file not found: {path}")
            continue

        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        if df.empty:
            continue

        if column is None:
            series = df.iloc[:, 0]
        elif isinstance(column, int):
            if column < 0 or column >= len(df.columns):
                print(f"Warning: column index {column} out of range for {path}")
                continue
            series = df.iloc[:, column]
        else:
            if column not in df.columns:
                print(f"Warning: column '{column}' not found in {path}")
                continue
            series = df[column]

        for raw in series.tolist():
            keyword = _th_normalize_ws(raw)
            if not keyword:
                continue
            key = keyword.lower()
            if key in seen:
                continue
            seen.add(key)
            loaded.append(keyword)

    return loaded


async def _th_collect_candidates(page, scroll_rounds: int, scroll_wait_ms: int) -> list[dict[str, str]]:
    seen: dict[str, dict[str, str]] = {}

    rounds = max(1, int(scroll_rounds))
    for _ in range(rounds):
        payload = await page.evaluate(
            '''() => {
                const rows = [];
                const anchors = Array.from(document.querySelectorAll('a[href*="/post/"]'));
                for (const anchor of anchors) {
                    const href = anchor.href || anchor.getAttribute('href') || '';
                    if (!href.includes('/post/')) continue;

                    const card =
                        anchor.closest('article, div[role="article"], div[data-pressable-container="true"]') ||
                        anchor.closest('div');

                    const timeEl = card ? card.querySelector('time') : null;
                    const datetime = timeEl ? (timeEl.getAttribute('datetime') || timeEl.textContent || '') : '';

                    const cardText = card ? (card.innerText || '') : '';

                    rows.push({
                        url: href,
                        datetime: datetime,
                        card_text: cardText,
                    });
                }
                return rows;
            }'''
        )

        for item in payload:
            url = _th_safe_text(item.get("url"))
            if not url:
                continue
            url = url.split("?")[0].split("#")[0]
            if url not in seen:
                seen[url] = {
                    "url": url,
                    "datetime": _th_safe_text(item.get("datetime")),
                    "card_text": _th_safe_text(item.get("card_text")),
                }

        await page.mouse.wheel(0, 2600)
        await page.wait_for_timeout(max(250, int(scroll_wait_ms)))

    return list(seen.values())


async def scrape_threads_keywords(
    keywords: list[str],
    max_posts_per_keyword: int = 30,
    scroll_rounds: int = 20,
    scroll_wait_ms: int = 1500,
    headless: bool = True,
    nav_timeout_ms: int = 60000,
    start_date: Any = None,
    end_date: Any = None,
) -> pd.DataFrame:
    columns = [
        "created_at",
        "id",
        "text",
        "keyword",
        "url",
        "author_username",
        "like_count",
        "comment_count",
        "share_count",
        "source",
        "scraped_at_utc",
    ]

    clean_keywords = [_th_normalize_ws(k) for k in keywords if _th_normalize_ws(k)]
    if not clean_keywords:
        return pd.DataFrame(columns=columns)

    start_ts = _th_to_utc_datetime(start_date)
    end_ts = _th_to_utc_datetime(end_date)

    records: list[dict[str, Any]] = []

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=headless)
        context = await browser.new_context(
            viewport={"width": 1280, "height": 2000},
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="en-US",
        )
        page = await context.new_page()
        page.set_default_timeout(int(nav_timeout_ms))

        try:
            for keyword in clean_keywords:
                print(f"\nScraping keyword: {keyword}")
                kept = 0
                per_keyword_seen: set[str] = set()

                search_url = f"https://www.threads.com/search?q={quote_plus(keyword)}"
                try:
                    await page.goto(search_url, wait_until="domcontentloaded")
                    await page.wait_for_timeout(1500)
                except PlaywrightTimeoutError:
                    print(f"  ! Timeout during search for keyword: {keyword}")
                    continue
                except Exception as exc:
                    print(f"  ! Failed to load keyword '{keyword}': {exc}")
                    continue

                candidates = await _th_collect_candidates(page, scroll_rounds=scroll_rounds, scroll_wait_ms=scroll_wait_ms)

                for item in candidates:
                    if kept >= int(max_posts_per_keyword):
                        break

                    post_url = _th_safe_text(item.get("url"))
                    if not post_url or post_url in per_keyword_seen:
                        continue
                    per_keyword_seen.add(post_url)

                    card_text = _th_normalize_ws(item.get("card_text"))
                    if not card_text:
                        continue

                    if keyword.lower() not in card_text.lower():
                        continue

                    created_ts = _th_to_utc_datetime(item.get("datetime"))
                    if start_ts is not None and (created_ts is None or created_ts < start_ts):
                        continue
                    if end_ts is not None and created_ts is not None and created_ts > end_ts:
                        continue

                    created_at = _th_to_unix_seconds(created_ts)
                    author_username = _th_extract_username(post_url)
                    post_id = _th_extract_post_id(post_url, card_text, created_at)

                    like_count = _th_extract_metric(card_text, ("likes", "like"))
                    comment_count = _th_extract_metric(card_text, ("replies", "reply", "comments", "comment"))
                    share_count = _th_extract_metric(card_text, ("reposts", "repost", "quotes", "quote", "shares", "share"))

                    records.append(
                        {
                            "created_at": created_at,
                            "id": post_id,
                            "text": card_text,
                            "keyword": keyword,
                            "url": post_url,
                            "author_username": author_username,
                            "like_count": like_count,
                            "comment_count": comment_count,
                            "share_count": share_count,
                            "source": "threads",
                            "scraped_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC"),
                        }
                    )
                    kept += 1

                print(f"  + Kept {kept} post(s)")
        finally:
            await context.close()
            await browser.close()

    if not records:
        return pd.DataFrame(columns=columns)

    out = pd.DataFrame(records)
    for col in columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[columns]


In [6]:
th_keywords = _th_load_keywords_from_files(TH_KEYWORD_FILES, TH_KEYWORD_COLUMNS)
if TH_MAX_KEYWORDS is not None:
    th_keywords = th_keywords[:TH_MAX_KEYWORDS]

print("Loaded keywords:", len(th_keywords))
print("Keyword sample:", th_keywords[:10])

threads_df = await scrape_threads_keywords(
    keywords=th_keywords,
    max_posts_per_keyword=TH_MAX_POSTS_PER_KEYWORD,
    scroll_rounds=TH_SCROLL_ROUNDS,
    scroll_wait_ms=TH_SCROLL_WAIT_MS,
    headless=TH_HEADLESS,
    nav_timeout_ms=TH_NAV_TIMEOUT_MS,
    start_date=TH_START_DATE,
    end_date=TH_END_DATE,
)

print("Rows collected:", len(threads_df))
threads_df.head()


Loaded keywords: 30
Keyword sample: ['biag a puyat', 'nabagbaga', 'nakaraot', 'nag-upa', 'agew ti puso', 'nagsalsal ti puso', 'pudot ti puso', 'nabati ti bagi', 'nag-init ti bagi', 'walay lakas']

Scraping keyword: biag a puyat
  + Kept 0 post(s)

Scraping keyword: nabagbaga
  + Kept 0 post(s)

Scraping keyword: nakaraot
  + Kept 0 post(s)

Scraping keyword: nag-upa
  + Kept 0 post(s)

Scraping keyword: agew ti puso
  + Kept 0 post(s)

Scraping keyword: nagsalsal ti puso
  + Kept 0 post(s)

Scraping keyword: pudot ti puso
  + Kept 0 post(s)

Scraping keyword: nabati ti bagi
  + Kept 0 post(s)

Scraping keyword: nag-init ti bagi
  + Kept 0 post(s)

Scraping keyword: walay lakas
  + Kept 0 post(s)

Scraping keyword: natandaan
  + Kept 0 post(s)

Scraping keyword: naglumpay
  + Kept 0 post(s)

Scraping keyword: nag-aalab
  + Kept 0 post(s)

Scraping keyword: nag-amis ti ngipan
  + Kept 0 post(s)

Scraping keyword: nagtarum
  + Kept 0 post(s)

Scraping keyword: nagatunanen
  + Kept 0 post(

,created_at,id,text,keyword,url,author_username,like_count,comment_count,share_count,source,scraped_at_utc
0,1772706783,b685375ddedc8e0f4db9a66110dbcdc459b6fd88,igorot.traveller\nIloilo City\n5d\nIf I had to...,hiligaynon,https://www.threads.com/@igorot.traveller/post...,,None,None,None,threads,2026-03-11 02:04:59 UTC
1,1771568968,264586f40e578922f4c9abb1a6ef5fb37b2010da,thecraftysol\n02/20/26\nThe Pitt is one absolu...,hiligaynon,https://www.threads.com/@thecraftysol/post/DU-...,,None,None,None,threads,2026-03-11 02:04:59 UTC
2,1771568968,583eb539c157f34602b87ba2bda6b74e3c96c310,thecraftysol\n02/20/26\nThe Pitt is one absolu...,hiligaynon,https://www.threads.com/@thecraftysol/post/DU-...,,None,None,None,threads,2026-03-11 02:04:59 UTC
3,1771641515,c7dd79c0d35d21393c139a5c4ad21355500a6919,talkintech\n02/21/26\nSome Pinoys are raising ...,hiligaynon,https://www.threads.com/@talkintech/post/DVAOQ...,,None,None,None,threads,2026-03-11 02:04:59 UTC
4,1771641515,275f9a47a6646162c1f69b6e2b907dfa213d2180,talkintech\n02/21/26\nSome Pinoys are raising ...,hiligaynon,https://www.threads.com/@talkintech/post/DVAOQ...,,None,None,None,threads,2026-03-11 02:04:59 UTC


In [7]:
TH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_core_cols = ["created_at", "id", "text"]
optional_cols = [
    "keyword",
    "url",
    "author_username",
    "like_count",
    "comment_count",
    "share_count",
    "source",
    "scraped_at_utc",
]

missing_core_cols = [col for col in required_core_cols if col not in threads_df.columns]
if missing_core_cols:
    raise ValueError(f"Missing required Threads columns before save: {missing_core_cols}")

ordered_optional = [col for col in optional_cols if col in threads_df.columns]
remaining_cols = [col for col in threads_df.columns if col not in required_core_cols + ordered_optional]
ordered_cols = required_core_cols + ordered_optional + remaining_cols

threads_to_save = threads_df[ordered_cols].copy()
before = len(threads_to_save)
threads_to_save = threads_to_save.drop_duplicates(subset=["id"], keep="first")
skipped_duplicates_within_run = int(before - len(threads_to_save))

file_exists = TH_OUTPUT_FILE.exists()
existing_ids: set[str] = set()

if file_exists:
    existing_header = pd.read_csv(TH_OUTPUT_FILE, nrows=0, encoding="utf-8-sig").columns.tolist()
    if "id" not in existing_header:
        raise ValueError(f"Existing Threads file is missing required 'id' column: {TH_OUTPUT_FILE}")
    existing_id_df = pd.read_csv(TH_OUTPUT_FILE, usecols=["id"], dtype=str, encoding="utf-8-sig")
    existing_ids = set(existing_id_df["id"].fillna("").astype(str).str.strip())
else:
    existing_header = ordered_cols

threads_to_save["_id_norm"] = threads_to_save["id"].fillna("").astype(str).str.strip()
to_append = threads_to_save[~threads_to_save["_id_norm"].isin(existing_ids)].drop(columns=["_id_norm"])
skipped_existing_rows = int(len(threads_to_save) - len(to_append))

for col in existing_header:
    if col not in to_append.columns:
        to_append[col] = pd.NA
extra_cols = [col for col in to_append.columns if col not in existing_header]
write_cols = existing_header + extra_cols

if len(to_append) > 0:
    to_append[write_cols].to_csv(
        TH_OUTPUT_FILE,
        mode="a" if file_exists else "w",
        header=not file_exists,
        index=False,
        encoding="utf-8-sig",
    )

print(f"Appended {len(to_append)} new row(s) -> {TH_OUTPUT_FILE}")
print(f"Skipped duplicates within run: {skipped_duplicates_within_run}")
print(f"Skipped already existing rows: {skipped_existing_rows}")


Appended 6 new row(s) -> /Users/angelodelapaz/Documents/GitHub/healthphpersonal/data/raw/threads/threads_notebook.csv
Skipped duplicates within run: 0
Skipped already existing rows: 12


In [8]:
required_cols = ["created_at", "id", "text"]
missing_cols = [col for col in required_cols if col not in threads_df.columns]
if missing_cols:
    raise ValueError(f"Missing required Threads columns: {missing_cols}")

created_series = threads_df["created_at"].fillna("").astype(str).str.strip()
validation_summary = pd.DataFrame({
    "metric": [
        "rows",
        "duplicate_id_rows",
        "missing_created_at",
        "missing_id",
        "missing_text",
        "unix10_created_at_rows",
    ],
    "value": [
        int(len(threads_df)),
        int(threads_df.duplicated(subset=["id"]).sum()) if not threads_df.empty else 0,
        int(created_series.eq("").sum()) if not threads_df.empty else 0,
        int(threads_df["id"].isna().sum() + (threads_df["id"].astype(str).str.strip() == "").sum()) if not threads_df.empty else 0,
        int(threads_df["text"].isna().sum() + (threads_df["text"].astype(str).str.strip() == "").sum()) if not threads_df.empty else 0,
        int(created_series.str.fullmatch(r"\d{10}", na=False).sum()) if not threads_df.empty else 0,
    ],
})

validation_summary

# Optional smoke test: verify merger can load Threads raw files
repo_root = TH_PROJECT_ROOT.resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.data_merger import load_source

threads_merged_preview = load_source("threads", data_root=repo_root / "data" / "raw")
print("Merged Threads rows available:", len(threads_merged_preview))
threads_merged_preview.head(3)


Loaded threads: 4,669 rows
Merged Threads rows available: 4669


,created_at,id,text,source
0,1771805019,3838501173108437570,I Beg of you to change the name of Usher syndr...,threads
1,1771804996,3838500978249470064,I Beg of you to change the name of Usher syndr...,threads
2,1771777886,3838273568287930837,Congress’ fate in 2027 was already on display ...,threads
